In [9]:
import sklearn
import pandas as pd
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import torch
import torchvision

In [10]:
device = torch.device("cuda")

In [11]:
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor




train_ds = MNIST(
    root="./data",
    download=True,
    train=True,
    transform=ToTensor(),
)

test_ds = MNIST(
    root="./data",
    download=True,
    train=False,
    transform=ToTensor()
)

In [12]:
from torch.utils.data import DataLoader

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=True)

In [13]:
from torch import nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 10)  # salida: 10 clases
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.fc(x)
        return logits  # logits para CrossEntropyLoss

In [14]:
mlp = MLP().to(device)
optimicer = torch.optim.AdamW(mlp.parameters())
loss_fn = nn.CrossEntropyLoss()


In [15]:
def train_model(model: nn.Module,
                loss_fn: nn.Module,
                optimizer: torch.optim.Optimizer,
                epochs: int = 5):


    best_loss = float('inf')
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            y_pred = model(X)
            loss_val = loss_fn(y_pred, y)
            loss_val.backward()
            optimizer.step()

            train_loss += loss_val.item() * X.size(0)

        train_loss /= len(train_dl.dataset)

        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in test_dl:
                X, y = X.to(device), y.to(device)
                y_pred = model(X)
                loss_val = loss_fn(y_pred, y)
                test_loss += loss_val.item() * X.size(0)

                preds = y_pred.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)

        test_loss /= len(test_dl.dataset)
        accuracy = correct / total

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Test Loss: {test_loss:.4f} | "
              f"Test Acc: {accuracy*100:.2f}%")

        # Guardar el mejor modelo según test loss
        if test_loss < best_loss:
            best_loss = test_loss
            best_model_state = model.state_dict()

    # Cargar mejor modelo
    model.load_state_dict(best_model_state)
    return model
    

In [16]:
best_mlp = train_model(mlp, loss_fn, optimicer, epochs=10)

Epoch 1/10 | Train Loss: 0.2280 | Test Loss: 0.1093 | Test Acc: 96.58%
Epoch 2/10 | Train Loss: 0.1034 | Test Loss: 0.0945 | Test Acc: 97.01%
Epoch 3/10 | Train Loss: 0.0818 | Test Loss: 0.0814 | Test Acc: 97.29%
Epoch 4/10 | Train Loss: 0.0700 | Test Loss: 0.0828 | Test Acc: 97.56%
Epoch 5/10 | Train Loss: 0.0608 | Test Loss: 0.0718 | Test Acc: 97.96%
Epoch 6/10 | Train Loss: 0.0520 | Test Loss: 0.0917 | Test Acc: 97.44%
Epoch 7/10 | Train Loss: 0.0479 | Test Loss: 0.0783 | Test Acc: 97.95%
Epoch 8/10 | Train Loss: 0.0447 | Test Loss: 0.0769 | Test Acc: 98.15%
Epoch 9/10 | Train Loss: 0.0407 | Test Loss: 0.0744 | Test Acc: 97.98%
Epoch 10/10 | Train Loss: 0.0385 | Test Loss: 0.0765 | Test Acc: 98.21%


In [17]:
img = cv.imread("received_image.jpg", cv.IMREAD_GRAYSCALE)
img = cv.resize(img, (28, 28), interpolation=cv.INTER_AREA)
img = cv.bitwise_not(img)
img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device) / 255.0
best_mlp.eval()
with torch.no_grad():
    output = best_mlp(img_tensor)
    probabilities = nn.Softmax(dim=1)(output)[0]
    print(probabilities.cpu().numpy())

[8.1423763e-04 5.1502633e-07 2.7774529e-05 8.4371413e-06 9.7808790e-01
 6.8292043e-06 5.1522831e-05 8.5415842e-05 3.5247682e-05 2.0882104e-02]


In [18]:
path = "models/pytorch/"

torch.save(best_mlp.state_dict(), path + "MultiLayerPerceptron.pth")

### CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # --- Bloques convolucionales ---
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),  # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28x28 -> 14x14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 14x14 -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14 -> 7x7

            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # 7x7 -> 7x7
            nn.ReLU(),
            nn.MaxPool2d(2)  # 7x7 -> 3x3
        )

        # --- Flatten + Fully Connected ---
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(128*3*3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)  # logits para 10 clases
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.flatten(x)
        logits = self.fc(x)
        return logits

In [ ]:
cnn = CNN().to(device)
optimizer = torch.optim.AdamW(cnn.parameters())
loss_fn = nn.CrossEntropyLoss()

In [21]:
best_cnn = train_model(cnn, loss_fn, optimizer, epochs=10)

Epoch 1/10 | Train Loss: 0.1767 | Test Loss: 0.0481 | Test Acc: 98.56%
Epoch 2/10 | Train Loss: 0.0530 | Test Loss: 0.0291 | Test Acc: 99.02%
Epoch 3/10 | Train Loss: 0.0386 | Test Loss: 0.0245 | Test Acc: 99.08%
Epoch 4/10 | Train Loss: 0.0308 | Test Loss: 0.0248 | Test Acc: 99.31%
Epoch 5/10 | Train Loss: 0.0234 | Test Loss: 0.0383 | Test Acc: 98.85%
Epoch 6/10 | Train Loss: 0.0208 | Test Loss: 0.0256 | Test Acc: 99.28%
Epoch 7/10 | Train Loss: 0.0172 | Test Loss: 0.0243 | Test Acc: 99.32%
Epoch 8/10 | Train Loss: 0.0158 | Test Loss: 0.0304 | Test Acc: 99.15%
Epoch 9/10 | Train Loss: 0.0137 | Test Loss: 0.0216 | Test Acc: 99.40%
Epoch 10/10 | Train Loss: 0.0116 | Test Loss: 0.0199 | Test Acc: 99.44%


In [22]:
img = cv.imread("received_image.jpg", cv.IMREAD_GRAYSCALE)
img = cv.resize(img, (28, 28), interpolation=cv.INTER_AREA)
img = cv.bitwise_not(img)
img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device) / 255.0
best_cnn.eval()
with torch.no_grad():
    output = best_cnn(img_tensor)
    probabilities = nn.Softmax(dim=1)(output)[0]
    print(probabilities.cpu().numpy())

[2.4381743e-11 1.6788377e-07 1.3095659e-08 1.5113219e-13 9.9999881e-01
 5.8576459e-09 3.8639109e-07 2.7429217e-09 3.0428843e-10 6.2807726e-07]


In [ ]:
path = "models/pytorch/"

torch.save(best_cnn.state_dict(), path + "ConvolutionalNeuralNetwork.pth")

: 